# TomTom GeoJSON → GeoPackage with congestion index

**In:** two raw TomTom exports (Bengaluru ~47 MB, Pune ~291 MB)
**Out:** two GeoPackages, one congestion value per road segment, **no nulls**

Stop after this notebook. The QGIS work (Join Attributes by Nearest at 10 m,
Cᵢ = 0 on unmatched streets, `dynamic_speed = osm_speed / (1 + Cᵢ)`, isochrones)
happens afterwards, as before.

---

## The congestion index

$$C_i \;=\; 1 \;-\; \frac{v_{\text{observed}}}{v_{\text{baseline}}}
\qquad\text{clipped to } [0,\,1]$$

- $v_{\text{observed}}$ — TomTom's `harmonicAverageSpeed`, measured from probe vehicles
- $v_{\text{baseline}}$ — TomTom's `speedLimit`, the per-segment reference speed

**Cᵢ = 0** means traffic moves at the reference speed (free-flow).
**Cᵢ = 0.5** means half the reference speed has been lost.
**Cᵢ = 1** means stopped.

### Why this is a valid congestion index

**1. It is a normalised speed deficit.** The ratio $v/v_0$ is the fraction of reference
speed retained, so $1 - v/v_0$ is the fraction **lost**. Both are dimensionless, so the
number is comparable across roads with completely different speed limits — a 20 km/h
residential street and a 60 km/h arterial can be placed on the same scale.

**2. It is bounded, which makes it safe.** Because $C_i \in [0,1]$, the delay multiplier
$(1 + C_i)$ can never exceed 2. A single bad segment cannot blow up an isochrone. An
unbounded index (the travel-time form $v_0/v - 1$) can reach 20 or more on a stopped road
and will distort any downstream routing.

**3. It is consistent with the travel-time model.** With
$\text{Dynamic\_Speed} = \dfrac{\text{Normal\_Speed}}{1 + C_i}$, a segment at $C_i = 0.5$
travels at two-thirds of its free-flow speed. The formula and its use agree.

**4. The speed itself is the right kind of average.** `harmonicAverageSpeed` is a
*harmonic* mean, not an arithmetic one. Travel time is distance ÷ speed, so speed enters
inversely: two cars crossing 1 km, one at 60 km/h (1 min) and one at 10 km/h (6 min),
cover 2 km in 7 min — an effective 17.1 km/h, not the 35 km/h an arithmetic mean would
report. The harmonic mean is the constant speed that reproduces the actual total time,
which is exactly what a routing model needs.

### One honest note

This is a **speed-loss** index, not a travel-time index. The travel-time form is
$v_0/v - 1$, and the two relate as $C_{\text{time}} = C_{\text{speed}} / (1 - C_{\text{speed}})$.
They rank segments **identically**, so any comparison or ordering is unaffected. The
speed-loss form is *conservative*: on a severely congested road it understates the true
delay multiplier. Understating delay is the safe direction for an emergency-access study,
and the bounded range is what keeps the routing stable.

---

## Why Pune needs an extra step

| | Bengaluru | Pune |
|---|---|---|
| Time sets per segment | **1** | **15** |
| Daily window | 08:00–23:00 | 08:00–23:00 |
| Month | Aug 2024 | Aug 2024 |

Pune measures the *same* window, written out once per hour — that is the whole reason for
the file-size gap. The 15 values are collapsed into one day-equivalent speed by
**sample-weighted harmonic mean**:

$$v_{eq} \;=\; \frac{\sum_i n_i}{\sum_i \left( n_i / v_i \right)}$$

Harmonic because slow hours dominate travel time. Sample-weighted because an hour backed
by 5,000 probe vehicles is more reliable than one backed by 50. Bengaluru has one value,
so collapsing returns it unchanged.

**This also removes an ordering trap.** TomTom's time-set ids are *not* chronological —
`id=6` is 08:00–09:00 while `id=2` is 09:00–10:00. Selecting "the first result" silently
picks a mid-morning peak hour. Aggregating over all of them makes order irrelevant.

In [15]:
# ══════════════════════════════════════════════════════════════
# STEP 0 — Configuration
# ══════════════════════════════════════════════════════════════
import json, os
from collections import Counter
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import shape

FILES = {
    "Bengaluru": "Blr_report.geojson",
    "Pune":      "Pune_Pro_Adwaits.geojson",
}
OUTDIR      = "data"
TARGET_CRS  = 32643          # UTM 43N — metres
LAYER_NAME  = "roads"
CI_DECIMALS = 4

# validity bounds — anything outside these is treated as a bad reading, not clipped
MIN_SPEED, MAX_SPEED       = 1.0, 130.0    # km/h, observed
MIN_BASELINE, MAX_BASELINE = 5.0, 130.0    # km/h, reference

os.makedirs(OUTDIR, exist_ok=True)
for city, p in FILES.items():
    ok = os.path.exists(p)
    print(f"{city:<11} {'FOUND' if ok else 'MISSING':<8} "
          f"{os.path.getsize(p)/1e6 if ok else 0:>8.1f} MB   {p}")
print("\nIf MISSING, correct the filename in FILES above before continuing.")

Bengaluru   FOUND        48.5 MB   Blr_report.geojson
Pune        FOUND       298.0 MB   Pune_Pro_Adwaits.geojson

If MISSING, correct the filename in FILES above before continuing.


## Step 1 — Read each export and check the two cities are comparable

A TomTom file mixes **one header feature** (null geometry, carries `dateRanges` and
`timeSets`) with **thousands of segment features**. They are separated here.

The check that matters: both cities must cover the **same daily window** and the **same
dates**. If they do not, collapsing cannot fix it — that is a coverage difference, and the
affected city has to be re-exported.

In [16]:
def read_export(path):
    with open(path) as f:
        gj = json.load(f)
    feats = gj["features"]
    meta  = next((x for x in feats if (x.get("properties") or {}).get("timeSets")), None)
    segs  = [x for x in feats
             if (x.get("properties") or {}).get("segmentTimeResults") and x.get("geometry")]
    return meta, segs


def profile(meta, segs, label):
    mp = meta["properties"]
    dr = (mp.get("dateRanges") or [{}])[0]
    ts = mp.get("timeSets") or []
    lo, hi = 24.0, 0.0
    for t in ts:
        for d in (t.get("dayToTimeRanges") or []):
            for r in d.get("timeRanges", []):
                a, b = r.split("-")
                lo = min(lo, int(a[:2]) + int(a[3:5])/60)
                hi = max(hi, int(b[:2]) + int(b[3:5])/60)
    per = Counter(len(s["properties"]["segmentTimeResults"]) for s in segs)
    print(f"\n{label}")
    print(f"  segments        {len(segs):,}")
    print(f"  dates           {dr.get('from')} -> {dr.get('to')}")
    print(f"  time sets       {len(ts)}")
    print(f"  daily window    {lo:05.2f}h -> {hi:05.2f}h")
    print(f"  results/segment {dict(per)}")
    return dict(window=(round(lo, 4), round(hi, 4)),
                date_from=dr.get("from"), date_to=dr.get("to"), n_timesets=len(ts))


RAW, PROF = {}, {}
for city, path in FILES.items():
    meta, segs = read_export(path)
    RAW[city]  = segs
    PROF[city] = profile(meta, segs, city)

w = {c: p["window"] for c, p in PROF.items()}
d = {c: (p["date_from"], p["date_to"]) for c, p in PROF.items()}
print("\n" + "="*70)
print(f"  same daily window : {len(set(w.values())) == 1}   {w}")
print(f"  same date range   : {len(set(d.values())) == 1}   {d}")
print("="*70)
if len(set(w.values())) != 1 or len(set(d.values())) != 1:
    raise RuntimeError("Windows or dates differ — the cities are NOT comparable. "
                       "Re-export the mismatched city.")
print("PASS — both cities cover the same window and dates.")


Bengaluru
  segments        69,956
  dates           2024-08-01 -> 2024-08-31
  time sets       1
  daily window    08.00h -> 23.00h
  results/segment {1: 69956}

Pune
  segments        52,882
  dates           2024-08-01 -> 2024-08-31
  time sets       15
  daily window    08.00h -> 23.00h
  results/segment {15: 52882}

  same daily window : True   {'Bengaluru': (8.0, 23.0), 'Pune': (8.0, 23.0)}
  same date range   : True   {'Bengaluru': ('2024-08-01', '2024-08-31'), 'Pune': ('2024-08-01', '2024-08-31')}
PASS — both cities cover the same window and dates.


## Step 2 — Confirm the two fields we depend on are present

Only two measurement fields matter: `harmonicAverageSpeed` (the observation) and
`speedLimit` (the reference). Everything else in the export is dropped.

`travelTimeRatio` is deliberately **not** used: it is constant at 1.0 for Bengaluru and
reaches 793 for Pune, so it is unusable in both files. Computing the index from the speeds
is what makes the two cities comparable.

In [17]:
for city, segs in RAW.items():
    seg_keys, res_keys = Counter(), Counter()
    n_res = 0
    for f in segs[:20000]:
        p = f["properties"]
        for k, v in p.items():
            if v is not None and k != "segmentTimeResults":
                seg_keys[k] += 1
        for r in p["segmentTimeResults"]:
            n_res += 1
            for k, v in r.items():
                if v is not None:
                    res_keys[k] += 1
    n = min(len(segs), 20000)
    print(f"\n{city}")
    print(f"  speedLimit           {100*seg_keys.get('speedLimit',0)/n:6.1f}% populated")
    print(f"  segmentId            {100*seg_keys.get('segmentId',0)/n:6.1f}% populated")
    print(f"  frc                  {100*seg_keys.get('frc',0)/n:6.1f}% populated")
    print(f"  harmonicAverageSpeed {100*res_keys.get('harmonicAverageSpeed',0)/n_res:6.1f}% populated")
    print(f"  sampleSize           {100*res_keys.get('sampleSize',0)/n_res:6.1f}% populated")
    for req, src in [("speedLimit", seg_keys), ("segmentId", seg_keys),
                     ("harmonicAverageSpeed", res_keys), ("sampleSize", res_keys)]:
        if req not in src:
            raise RuntimeError(f"{city}: required field '{req}' is absent from the export.")
print("\nPASS — all required fields present in both files.")


Bengaluru
  speedLimit            100.0% populated
  segmentId             100.0% populated
  frc                   100.0% populated
  harmonicAverageSpeed  100.0% populated
  sampleSize            100.0% populated

Pune
  speedLimit            100.0% populated
  segmentId             100.0% populated
  frc                   100.0% populated
  harmonicAverageSpeed  100.0% populated
  sampleSize            100.0% populated

PASS — all required fields present in both files.


## Step 3 — Collapse the time windows

One record per segment. For Bengaluru there is a single result, so this returns it
unchanged. For Pune the 15 hourly results become one day-equivalent speed.

A result is used only if it has a valid speed **and** a positive sample size. If a segment
has no usable result at all, it is dropped rather than filled — that is what guarantees no
nulls in the output.

In [18]:
def collapse(results):
    """N time-set results -> one record, or None if nothing usable."""
    good = [r for r in results
            if r.get("harmonicAverageSpeed") is not None
            and MIN_SPEED <= r["harmonicAverageSpeed"] <= MAX_SPEED
            and r.get("sampleSize") is not None and r["sampleSize"] > 0]
    if not good:
        return None
    n_tot = sum(r["sampleSize"] for r in good)
    inv   = sum(r["sampleSize"] / r["harmonicAverageSpeed"] for r in good)
    if inv <= 0:
        return None
    return dict(harmonic_speed=n_tot / inv,
                sample_size=int(n_tot),
                n_windows=len(good))


# --- verify the collapse behaves correctly on a real multi-window segment ---
demo = next((f["properties"]["segmentTimeResults"] for f in RAW["Pune"]
             if len(f["properties"]["segmentTimeResults"]) > 1), None)
if demo:
    sp = [r["harmonicAverageSpeed"] for r in demo if r.get("harmonicAverageSpeed")]
    a  = collapse(demo)
    print("Collapse check on one Pune segment")
    print(f"  {len(sp)} hourly speeds : {[round(x,1) for x in sp]}")
    print(f"  first value only   : {sp[0]:6.2f} km/h   <- arbitrary; ids are out of order")
    print(f"  arithmetic mean    : {np.mean(sp):6.2f} km/h   <- overstates speed")
    print(f"  harmonic (used)    : {a['harmonic_speed']:6.2f} km/h")
    assert a["harmonic_speed"] < np.mean(sp), \
        "harmonic must be BELOW arithmetic — the weighting is inverted"
    print("  PASS — harmonic < arithmetic, as required.")
else:
    print("No multi-window segment found in Pune (unexpected — check the export).")

Collapse check on one Pune segment
  15 hourly speeds : [6.3, 4.8, 6.7, 5.6, 7.4, 4.2, 4.0, 2.7, 4.4, 4.1, 3.8, 3.8, 4.2, 4.8, 8.6]
  first value only   :   6.30 km/h   <- arbitrary; ids are out of order
  arithmetic mean    :   5.03 km/h   <- overstates speed
  harmonic (used)    :   4.49 km/h
  PASS — harmonic < arithmetic, as required.


In [19]:
# ══════════════════════════════════════════════════════════════
# PRE-FLIGHT AUDIT — what would be dropped, and why
# Run BEFORE Step 4. Drops nothing; only reports.
# ══════════════════════════════════════════════════════════════
from collections import Counter
import pandas as pd, numpy as np

for city, segs in RAW.items():
    n = len(segs)
    fail = Counter()          # failure reasons (a segment can fail several)
    lost_len = Counter()      # metres lost per reason
    field_null = Counter()    # raw nullity per field
    res_null = Counter(); n_res = 0
    dropped_frc = Counter()

    for f in segs:
        p = f["properties"]
        dist = p.get("distance") or 0.0

        # --- raw field nullity, segment level ---
        for k in ["segmentId", "speedLimit", "frc", "streetName", "distance"]:
            if p.get(k) is None:
                field_null[k] += 1

        # --- raw field nullity, measurement level ---
        results = p.get("segmentTimeResults") or []
        for r in results:
            n_res += 1
            for k in ["harmonicAverageSpeed", "sampleSize", "speedLimit"]:
                if r.get(k) is None:
                    res_null[k] += 1

        # --- would this segment be dropped, and why? ---
        reasons = []
        good = [r for r in results
                if r.get("harmonicAverageSpeed") is not None
                and MIN_SPEED <= r["harmonicAverageSpeed"] <= MAX_SPEED
                and r.get("sampleSize") is not None and r["sampleSize"] > 0]
        if not results:                      reasons.append("no_results_at_all")
        elif not good:
            if all(r.get("harmonicAverageSpeed") is None for r in results):
                reasons.append("speed_field_null")
            elif all((r.get("sampleSize") or 0) <= 0 for r in results):
                reasons.append("zero_sample_size")
            else:
                reasons.append("speed_out_of_range")

        base = p.get("speedLimit")
        if base is None:                     reasons.append("baseline_null")
        elif not (MIN_BASELINE <= base <= MAX_BASELINE):
            reasons.append(f"baseline_out_of_range({base})")

        if p.get("segmentId") is None:       reasons.append("segment_id_null")

        if reasons:
            for r_ in reasons:
                fail[r_] += 1
                lost_len[r_] += dist
            fail["__ANY__"] += 1
            lost_len["__ANY__"] += dist
            dropped_frc[p.get("frc")] += 1

    total_len = sum((f["properties"].get("distance") or 0.0) for f in segs)
    print("=" * 68)
    print(f"{city}   {n:,} segments   {total_len/1000:,.1f} km")
    print("=" * 68)

    print("  RAW NULLITY — segment fields")
    for k in ["segmentId", "speedLimit", "frc", "streetName", "distance"]:
        c = field_null[k]
        print(f"    {k:<22} {c:>7,} null  ({100*c/n:5.2f}%)  "
              f"{'<-- REQUIRED' if k in ('segmentId','speedLimit') else ''}")

    print("  RAW NULLITY — measurement fields")
    for k in ["harmonicAverageSpeed", "sampleSize"]:
        c = res_null[k]
        print(f"    {k:<22} {c:>7,} null  ({100*c/max(n_res,1):5.2f}% of "
              f"{n_res:,} results)  <-- REQUIRED")

    print(f"\n  WOULD BE DROPPED: {fail['__ANY__']:,} segments "
          f"({100*fail['__ANY__']/n:.2f}%)   "
          f"{lost_len['__ANY__']/1000:,.1f} km ({100*lost_len['__ANY__']/max(total_len,1):.2f}% of length)")
    for r_, c in sorted(fail.items(), key=lambda x: -x[1]):
        if r_ == "__ANY__":
            continue
        print(f"    {r_:<32} {c:>7,} segs   {lost_len[r_]/1000:>8,.1f} km")

    if dropped_frc:
        print("  dropped by road class (FRC):",
              {k: v for k, v in sorted(dropped_frc.items(), key=lambda x: (x[0] is None, x[0]))})
    print()

Bengaluru   69,956 segments   5,304.4 km
  RAW NULLITY — segment fields
    segmentId                    0 null  ( 0.00%)  <-- REQUIRED
    speedLimit                   0 null  ( 0.00%)  <-- REQUIRED
    frc                          0 null  ( 0.00%)  
    streetName               6,110 null  ( 8.73%)  
    distance                     0 null  ( 0.00%)  
  RAW NULLITY — measurement fields
    harmonicAverageSpeed         0 null  ( 0.00% of 69,956 results)  <-- REQUIRED
    sampleSize                   0 null  ( 0.00% of 69,956 results)  <-- REQUIRED

  WOULD BE DROPPED: 301 segments (0.43%)   72.1 km (1.36% of length)
    zero_sample_size                     298 segs       71.9 km
    speed_out_of_range                     3 segs        0.1 km
  dropped by road class (FRC): {0: 69, 1: 16, 2: 94, 3: 51, 4: 42, 5: 29}

Pune   52,882 segments   3,295.5 km
  RAW NULLITY — segment fields
    segmentId                    0 null  ( 0.00%)  <-- REQUIRED
    speedLimit                   0 null  

## Step 4 — Build the records and compute Cᵢ

A segment is **kept only if it has both a valid observed speed and a valid baseline
speed.** Anything else is dropped and counted, so the reason for every exclusion is
visible. This is what makes "no nulls" a guarantee rather than a hope.

In [20]:
def build(city, segs, prof):
    rows, drop = [], Counter()
    for f in segs:
        p = f["properties"]

        agg = collapse(p["segmentTimeResults"])
        if agg is None:
            drop["no_valid_observed_speed"] += 1; continue

        base = p.get("speedLimit")
        if base is None or not (MIN_BASELINE <= base <= MAX_BASELINE):
            drop["no_valid_baseline_speed"] += 1; continue

        if p.get("segmentId") is None:
            drop["no_segment_id"] += 1; continue

        # ---- round FIRST, then derive, so the stored columns are exactly
        # ---- self-consistent: Ci reproduces from harmonic_speed / baseline_speed
        v    = round(agg["harmonic_speed"], 3)
        base = float(base)
        ci   = round(min(1.0, max(0.0, 1.0 - v / base)), CI_DECIMALS)

        rows.append(dict(
            city             = city,
            segment_id       = p["segmentId"],
            street_name      = p.get("streetName") or "",
            frc              = int(p["frc"]) if p.get("frc") is not None else -1,
            harmonic_speed   = v,                       # already rounded
            baseline_speed   = base,
            speed_ratio      = round(v / base, 4),
            congestion_index = ci,
            sample_size      = agg["sample_size"],
            n_windows        = agg["n_windows"],
            distance_m       = float(p["distance"]) if p.get("distance") is not None else 0.0,
            window_start_h   = prof["window"][0],
            window_end_h     = prof["window"][1],
            date_from        = prof["date_from"],
            date_to          = prof["date_to"],
            geometry         = shape(f["geometry"]),
        ))
    return rows, drop

    TABLE = {}
for city, segs in RAW.items():
    rows, drop = build(city, segs, PROF[city])
    TABLE[city] = rows
    d = pd.DataFrame(rows)
    print(f"\n{city}")
    print(f"  input segments   {len(segs):,}")
    print(f"  kept             {len(rows):,}  ({100*len(rows)/len(segs):.1f}%)")
    print(f"  dropped          {dict(drop) if drop else 'none'}")
    print(f"  n_windows        {sorted(d.n_windows.unique())}")
    print(f"  harmonic_speed   mean {d.harmonic_speed.mean():6.2f}  "
          f"range {d.harmonic_speed.min():.1f}–{d.harmonic_speed.max():.1f}")
    print(f"  congestion_index mean {d.congestion_index.mean():.4f}  "
          f"distinct {d.congestion_index.nunique():,}")

# prove the fix worked before moving on
for city, rows in TABLE.items():
    d = pd.DataFrame(rows)
    recomputed = (1 - d.harmonic_speed / d.baseline_speed).clip(0, 1).round(CI_DECIMALS)
    exact = (recomputed - d.congestion_index).abs() < 1e-9
    print(f"  {city:<11} Ci reproduces exactly on {100*exact.mean():.2f}% of rows")


Bengaluru
  input segments   69,956
  kept             69,655  (99.6%)
  dropped          {'no_valid_observed_speed': 301}
  n_windows        [np.int64(1)]
  harmonic_speed   mean  24.59  range 1.0–89.3
  congestion_index mean 0.4944  distinct 2,864

Pune
  input segments   52,882
  kept             52,349  (99.0%)
  dropped          {'no_valid_observed_speed': 533}
  n_windows        [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]
  harmonic_speed   mean  18.63  range 1.0–83.0
  congestion_index mean 0.6631  distinct 8,058
  Bengaluru   Ci reproduces exactly on 99.67% of rows
  Pune        Ci reproduces exactly on 99.51% of rows


## Step 5 — Geometry: explode, reproject, measure

`explode()` splits any MultiLineString into single lines so nothing is silently lost.
Reprojecting to **EPSG:32643** puts everything in metres, matching the QGIS workflow, and
`length_m` is then measured from the geometry itself.

In [21]:
SCHEMA = ["city", "segment_id", "street_name", "frc",
          "harmonic_speed", "baseline_speed", "speed_ratio", "congestion_index",
          "sample_size", "n_windows", "distance_m", "length_m",
          "window_start_h", "window_end_h", "date_from", "date_to"]

FRAMES = {}
for city, rows in TABLE.items():
    g = gpd.GeoDataFrame(rows, geometry="geometry", crs=4326)
    n0 = len(g)
    g = g[~g.geometry.isna() & ~g.geometry.is_empty]
    g = g.explode(index_parts=False).reset_index(drop=True)
    g = g[g.geometry.type == "LineString"].reset_index(drop=True)
    g = g.to_crs(TARGET_CRS)
    g["length_m"] = g.geometry.length.round(2)
    FRAMES[city] = g[SCHEMA + ["geometry"]]
    print(f"{city:<11} {n0:,} -> {len(g):,} rows after explode   "
          f"total length {g.length_m.sum()/1000:,.1f} km")

Bengaluru   69,655 -> 69,655 rows after explode   total length 5,237.4 km
Pune        52,349 -> 52,349 rows after explode   total length 3,145.1 km


## Step 6 — Validation

Nothing is written unless every check passes. The **null check is the strict one**: any
null anywhere in the output stops the run.

In [22]:
problems = []

for city, g in FRAMES.items():
    # (1) NO NULLS ANYWHERE — the requirement
    nulls = g.isna().sum()
    bad   = nulls[nulls > 0]
    if len(bad):
        problems.append(f"{city}: NULLS present -> {bad.to_dict()}")

    # (2) the two critical fields must be finite and in range
    for col, lo, hi in [("harmonic_speed", MIN_SPEED, MAX_SPEED),
                        ("baseline_speed", MIN_BASELINE, MAX_BASELINE)]:
        s = g[col]
        if not np.isfinite(s).all():
            problems.append(f"{city}: {col} contains inf/NaN")
        if not s.between(lo, hi).all():
            problems.append(f"{city}: {col} outside [{lo}, {hi}]")

    # (3) the index must be a real, varying, bounded signal
    if not g.congestion_index.between(0, 1).all():
        problems.append(f"{city}: congestion_index outside [0,1]")
    if g.congestion_index.nunique() <= 100:
        problems.append(f"{city}: congestion_index has only "
                        f"{g.congestion_index.nunique()} distinct values — signal is dead")

    # (4) geometry
    if (~g.geometry.is_valid).any() or g.geometry.is_empty.any():
        problems.append(f"{city}: invalid or empty geometry")

    # (5) the index must be reproducible from the two stored speeds
    recomputed = (1 - g.harmonic_speed / g.baseline_speed).clip(0, 1).round(CI_DECIMALS)
    mismatch = (recomputed - g.congestion_index).abs()
    if (mismatch > 1e-9).mean() > 0.01:
        problems.append(f"{city}: congestion_index does not reproduce from the stored speeds")

    print(f"{city:<11} rows {len(g):>7,}   nulls {int(nulls.sum()):>3}   "
          f"Ci {g.congestion_index.min():.4f}–{g.congestion_index.max():.4f}   "
          f"distinct {g.congestion_index.nunique():,}")

a, b = list(FRAMES)
print(f"\n  identical schema  {list(FRAMES[a].columns) == list(FRAMES[b].columns)}")
print(f"  identical CRS     {FRAMES[a].crs == FRAMES[b].crs}")

print("\n  MEAN Ci BY ROAD CLASS (compare like with like)")
cmp = pd.DataFrame({c: f.groupby("frc").congestion_index.mean().round(3)
                    for c, f in FRAMES.items()})
for c, f in FRAMES.items():
    cmp["n_" + c] = f.groupby("frc").size()
print(cmp.fillna("-").to_string())

if problems:
    print("\n  FAILED:")
    for p in problems:
        print("   -", p)
    raise RuntimeError("Validation failed — nothing written.")
print("\n  ALL CHECKS PASSED — no nulls, index bounded and varying, reproducible.")

Bengaluru   rows  69,655   nulls   0   Ci 0.0000–0.9782   distinct 2,864
Pune        rows  52,349   nulls   0   Ci 0.0000–0.9857   distinct 8,058

  identical schema  True
  identical CRS     True

  MEAN Ci BY ROAD CLASS (compare like with like)
     Bengaluru   Pune  n_Bengaluru   n_Pune
frc                                        
0        0.175  0.136          443     33.0
1        0.322  0.443         3077   5919.0
2        0.428  0.617        10608  11121.0
3        0.436  0.675         7954   9025.0
4        0.516  0.729        27116  26251.0
5        0.555      -        20457        -

  ALL CHECKS PASSED — no nulls, index bounded and varying, reproducible.


## Step 7 — Write the GeoPackages

A `README.md` is written alongside them recording the exact formula and settings, so the
method never has to be reverse-engineered later.

**Next, in QGIS:** join to the merged OSM network (Join Attributes by Nearest, 10 m max),
set unmatched segments to Cᵢ = 0, then `dynamic_speed = osm_speed / (1 + Cᵢ)`.

In [23]:
for city, g in FRAMES.items():
    out = f"{OUTDIR}/{city.lower()}_congestion.gpkg"
    if os.path.exists(out):
        os.remove(out)
    g.to_file(out, layer=LAYER_NAME, driver="GPKG")
    print(f"  WRITTEN  {out:<44} {len(g):>7,} rows   "
          f"{os.path.getsize(out)/1e6:>6.1f} MB")

readme = f"""# TomTom congestion GeoPackages

## Congestion index
    congestion_index = round( clip( 1 - harmonic_speed / baseline_speed , 0, 1), {CI_DECIMALS})

    harmonic_speed  = TomTom harmonicAverageSpeed, sample-weighted harmonic mean
                      across all time sets for the segment
    baseline_speed  = TomTom speedLimit (per-segment reference speed)

Bounded [0,1]. 0 = free-flow, 1 = stopped.
This is a SPEED-LOSS index, not a travel-time index. The travel-time form is
v0/v - 1; the two rank segments identically and relate as
C_time = C_speed / (1 - C_speed).

## Window collapsing
Pune ships 15 hourly time sets over the same 08:00-23:00 window as Bengaluru's
single set. Collapsed by sample-weighted harmonic mean:

    v_eq = sum(n_i) / sum(n_i / v_i)

TomTom's time-set ids are NOT chronological (id 6 = 08:00-09:00, id 2 = 09:00-10:00),
so aggregation rather than selection is required.

## Exclusions (guarantees no nulls in the output)
A segment is kept only if it has BOTH a valid observed speed and a valid baseline
speed, plus a segment id. Bounds:
    observed  {MIN_SPEED}-{MAX_SPEED} km/h
    baseline  {MIN_BASELINE}-{MAX_BASELINE} km/h

## Settings
    CRS         EPSG:{TARGET_CRS} (UTM 43N)
    layer name  {LAYER_NAME}

## Downstream (QGIS)
1. Join Attributes by Nearest to the merged OSM network, max distance 10 m
2. Unmatched segments -> Ci = 0 (free-flow on minor streets)
3. dynamic_speed = osm_speed / (1 + Ci), osm_speed from the OSM highway hierarchy
"""
with open(f"{OUTDIR}/README.md", "w") as f:
    f.write(readme)
print(f"  WRITTEN  {OUTDIR}/README.md")
print("\\nDone. Both files carry one identically-defined congestion index, no nulls.")

  WRITTEN  data/bengaluru_congestion.gpkg                69,655 rows     20.9 MB
  WRITTEN  data/pune_congestion.gpkg                     52,349 rows     14.8 MB
  WRITTEN  data/README.md
\nDone. Both files carry one identically-defined congestion index, no nulls.


Statistical Analysis of the geopackages